# 👕 Lookzi — Virtual Try-On

> **Avval:** `Runtime → Change runtime type → T4 GPU` tanlang, keyin `Ctrl+F9`

| Qadam | Vaqt |
|-------|------|
| 1. GPU check + install | ~2 min |
| 2. Clone repo | ~10 sec |
| 3. Model download (~27 GB) | ~20-30 min |
| 4. Load models | ~3 min |
| 5. Launch app | instant |
| Inference | ~1-2 min |

In [ ]:
# 1. GPU tekshirish va kutubxonalar o'rnatish
import subprocess, sys, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'GPU topilmadi! Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

pkgs = [
    'diffusers==0.27.2', 'transformers==4.46.3', 'accelerate==1.1.1',
    'huggingface_hub==0.26.5', 'gradio==4.44.0', 'onnxruntime',
    'einops', 'opencv-python-headless', 'scipy',
]
for p in pkgs:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', p], check=True)
print('\u2705 Packages installed')


In [ ]:
# 2. Repo clone
import os
REPO = '/content/Smart-mirror-MVP'
if not os.path.exists(REPO):
    os.system('git clone https://github.com/Mohamed-Kudratov/Smart-mirror-MVP.git ' + REPO)
    print('\u2705 Repo cloned')
else:
    os.system(f'cd {REPO} && git pull')
    print('\u2705 Repo updated')


In [ ]:
# 3. IDM-VTON modellarini yuklab olish (~27 GB)
# Keyingi sessiyalarda Drive ishlatish:
#   from google.colab import drive; drive.mount('/content/drive')
#   MODEL_DIR = '/content/drive/MyDrive/IDM-VTON-models'
from huggingface_hub import snapshot_download
import os

MODEL_DIR = '/content/IDM-VTON-models'
if not os.path.exists(os.path.join(MODEL_DIR, 'unet')):
    os.makedirs(MODEL_DIR, exist_ok=True)
    print('Downloading (~27 GB) — 20-30 daqiqa...')
    snapshot_download(
        repo_id='yisol/IDM-VTON',
        local_dir=MODEL_DIR,
        ignore_patterns=['*.md', '*.txt', '.gitattributes'],
    )
    print('\u2705 Models downloaded!')
else:
    print('\u2705 Models already present')
print('Subfolders:', os.listdir(MODEL_DIR))


In [ ]:
# 4. Modellarni yuklash
import os, sys, warnings, torch
warnings.filterwarnings('ignore')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

REPO      = '/content/Smart-mirror-MVP'
MODEL_DIR = '/content/IDM-VTON-models'
DEVICE    = 'cuda'
DTYPE     = torch.float16
W, H      = 768, 1024

for p in [REPO, f'{REPO}/src', f'{REPO}/preprocess',
          f'{REPO}/preprocess/openpose',
          f'{REPO}/preprocess/openpose/annotator']:
    if p not in sys.path: sys.path.insert(0, p)

import annotator.util as _au
_au.annotator_ckpts_path = os.path.join(MODEL_DIR, 'openpose', 'ckpts')

import numpy as np, onnxruntime as ort
from PIL import Image
from torchvision import transforms
from torchvision.transforms.functional import to_pil_image
from transformers import (CLIPImageProcessor, CLIPVisionModelWithProjection,
    CLIPTextModel, CLIPTextModelWithProjection, AutoTokenizer)
from diffusers import DDPMScheduler, AutoencoderKL
from unet_hacked_tryon   import UNet2DConditionModel
from unet_hacked_garmnet import UNet2DConditionModel as UNet2DConditionModel_ref
from tryon_pipeline      import StableDiffusionXLInpaintPipeline as TryonPipeline
from preprocess.openpose.run_openpose    import OpenPose
from preprocess.humanparsing.parsing_api import onnx_inference
from utils_mask import get_mask_location

print('Loading models...')

class Parsing:
    def __init__(self):
        opts = ort.SessionOptions()
        self.session     = ort.InferenceSession(
            os.path.join(MODEL_DIR,'humanparsing','parsing_atr.onnx'),
            sess_options=opts, providers=['CPUExecutionProvider'])
        self.lip_session = ort.InferenceSession(
            os.path.join(MODEL_DIR,'humanparsing','parsing_lip.onnx'),
            sess_options=opts, providers=['CPUExecutionProvider'])
    def __call__(self, img): return onnx_inference(self.session, self.lip_session, img)

parsing_model  = Parsing()
openpose_model = OpenPose(0)
print('  \u2705 OpenPose + Parsing')

def load_densepose():
    cache = '/content/densepose_cache'
    os.makedirs(cache, exist_ok=True)
    path = os.path.join(cache, 'densepose_r50_fpn_dl.torchscript')
    if not os.path.exists(path):
        print('  Downloading DensePose (~250 MB)...')
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(
            repo_id='LayerNorm/DensePose-TorchScript-with-hint-image',
            filename='densepose_r50_fpn_dl.torchscript', local_dir=cache)
    return torch.jit.load(path, map_location='cpu')

densepose_model = load_densepose()
densepose_model.eval()
print('  \u2705 DensePose')

unet = UNet2DConditionModel.from_pretrained(
    MODEL_DIR, subfolder='unet', torch_dtype=DTYPE).requires_grad_(False).eval()
unet_encoder = UNet2DConditionModel_ref.from_pretrained(
    MODEL_DIR, subfolder='unet_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
vae = AutoencoderKL.from_pretrained(
    MODEL_DIR, subfolder='vae', torch_dtype=DTYPE).requires_grad_(False).eval()
text_enc1 = CLIPTextModel.from_pretrained(
    MODEL_DIR, subfolder='text_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
text_enc2 = CLIPTextModelWithProjection.from_pretrained(
    MODEL_DIR, subfolder='text_encoder_2', torch_dtype=DTYPE).requires_grad_(False).eval()
img_enc = CLIPVisionModelWithProjection.from_pretrained(
    MODEL_DIR, subfolder='image_encoder', torch_dtype=DTYPE).requires_grad_(False).eval()
tok1  = AutoTokenizer.from_pretrained(MODEL_DIR, subfolder='tokenizer',   use_fast=False)
tok2  = AutoTokenizer.from_pretrained(MODEL_DIR, subfolder='tokenizer_2', use_fast=False)
sched = DDPMScheduler.from_pretrained(MODEL_DIR, subfolder='scheduler')
print('  \u2705 UNet, VAE, encoders')

pipe = TryonPipeline.from_pretrained(
    MODEL_DIR, unet=unet, vae=vae, feature_extractor=CLIPImageProcessor(),
    text_encoder=text_enc1, text_encoder_2=text_enc2,
    tokenizer=tok1, tokenizer_2=tok2,
    scheduler=sched, image_encoder=img_enc, torch_dtype=DTYPE)
pipe.unet_encoder = unet_encoder
pipe.to(DEVICE)
pipe.unet_encoder.to(DEVICE)
densepose_model.to(DEVICE)
pipe.enable_attention_slicing(1)
pipe.enable_vae_slicing()

tensor_tf = transforms.Compose([
    transforms.ToTensor(), transforms.Normalize([0.5],[0.5])])

used  = torch.cuda.memory_allocated()/1024**3
total = torch.cuda.get_device_properties(0).total_memory/1024**3
print(f'\n\u2705 All models ready!  VRAM: {used:.1f} / {total:.1f} GB')


In [ ]:
# 5. Gradio app
import gradio as gr, gc

def get_pose_image(person_pil):
    import cv2
    from einops import rearrange
    img_rgb = np.array(person_pil.resize((W,H)).convert('RGB'))
    inp = rearrange(torch.from_numpy(img_rgb).float(),'h w c -> c h w')
    canvas = np.zeros((H,W,3), dtype=np.uint8)
    with torch.no_grad():
        pred_boxes,_,fine_segm,_,_ = densepose_model(inp.to(DEVICE))
    for i in range(len(pred_boxes)):
        x1,y1,x2,y2 = [int(c) for c in pred_boxes[i].tolist()]
        x1,y1,x2,y2 = max(0,x1),max(0,y1),min(W-1,x2),min(H-1,y2)
        bw,bh = x2-x1,y2-y1
        if bw<=0 or bh<=0: continue
        parts = fine_segm[i].argmax(0).cpu().numpy().astype(np.uint8)
        parts = cv2.resize(parts,(bw,bh),interpolation=cv2.INTER_NEAREST)
        col = cv2.applyColorMap((parts*10).clip(0,255).astype(np.uint8),cv2.COLORMAP_VIRIDIS)
        canvas[y1:y2,x1:x2] = cv2.cvtColor(col,cv2.COLOR_BGR2RGB)
    return Image.fromarray(canvas)

def run_tryon(person_np, garment_np, garment_desc,
              auto_mask, steps, seed, progress=gr.Progress()):
    if person_np is None or garment_np is None:
        return None, None, 'Person va garment rasmlarini yuklang.'
    try:
        person  = Image.fromarray(person_np).convert('RGB').resize((W,H))
        garment = Image.fromarray(garment_np).convert('RGB').resize((W,H))
        if auto_mask:
            progress(0.10,'OpenPose...')
            keypoints = openpose_model(person.resize((384,512)))
            progress(0.20,'Human parsing...')
            parse_result,_ = parsing_model(person.resize((384,512)))
            progress(0.30,'Building mask...')
            mask,_ = get_mask_location('hd','upper_body',parse_result,keypoints)
            mask = mask.resize((W,H))
        else:
            from PIL import ImageDraw
            mask = Image.new('L',(W,H),0)
            ImageDraw.Draw(mask).rectangle(
                [int(W*0.05),int(H*0.10),int(W*0.95),int(H*0.72)],fill=255)
        mask_gray_t  = (1-tensor_tf(mask))*tensor_tf(person)
        mask_preview = to_pil_image(((mask_gray_t+1.0)/2.0).clamp(0,1))
        progress(0.40,'DensePose...')
        pose_img = get_pose_image(person)
        progress(0.50,'Encoding prompts...')
        neg = 'monochrome, lowres, bad anatomy, worst quality, low quality'
        with torch.no_grad():
            p_emb,n_emb,p_pool,n_pool = pipe.encode_prompt(
                f'model is wearing {garment_desc}',
                num_images_per_prompt=1, do_classifier_free_guidance=True,
                negative_prompt=neg)
            c_emb,_,_,_ = pipe.encode_prompt(
                [f'a photo of {garment_desc}'],
                num_images_per_prompt=1, do_classifier_free_guidance=False,
                negative_prompt=[''])
        progress(0.60,f'Diffusion ({int(steps)} steps)...')
        pose_t    = tensor_tf(pose_img).unsqueeze(0).to(DEVICE,DTYPE)
        garment_t = tensor_tf(garment).unsqueeze(0).to(DEVICE,DTYPE)
        gen = torch.Generator(DEVICE).manual_seed(int(seed))
        with torch.no_grad(), torch.cuda.amp.autocast(), torch.inference_mode():
            images = pipe(
                prompt_embeds=p_emb.to(DEVICE,DTYPE),
                negative_prompt_embeds=n_emb.to(DEVICE,DTYPE),
                pooled_prompt_embeds=p_pool.to(DEVICE,DTYPE),
                negative_pooled_prompt_embeds=n_pool.to(DEVICE,DTYPE),
                num_inference_steps=int(steps), generator=gen, strength=1.0,
                pose_img=pose_t, text_embeds_cloth=c_emb.to(DEVICE,DTYPE),
                cloth=garment_t, mask_image=mask, image=person,
                height=H, width=W, ip_adapter_image=garment,
                guidance_scale=2.0)[0]
        torch.cuda.empty_cache()
        return images[0], mask_preview, 'Done!'
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        return None, None, 'GPU out of memory - try fewer steps'
    except Exception as e:
        import traceback
        return None, None, f'Error: {e}\n{traceback.format_exc()}'
    finally:
        torch.cuda.empty_cache(); gc.collect()

REPO = '/content/Smart-mirror-MVP'
def _imgs(folder):
    if not os.path.exists(folder): return []
    return sorted([f'{folder}/{f}' for f in os.listdir(folder)
                   if f.lower().endswith(('.jpg','.jpeg','.png'))])

with gr.Blocks(title='Lookzi Virtual Try-On', theme=gr.themes.Soft()) as demo:
    gr.Markdown('# 👕 Lookzi — Virtual Try-On\nRasm yuklang va **Try It On** bosing.')
    with gr.Row():
        with gr.Column():
            person_in = gr.Image(label='Person Photo', type='numpy', height=420)
            gr.Examples(_imgs(f'{REPO}/example/human'), inputs=person_in,
                        examples_per_page=4, label='Sample people')
        with gr.Column():
            garment_in = gr.Image(label='Garment Photo', type='numpy', height=420)
            gr.Examples(_imgs(f'{REPO}/example/cloth'), inputs=garment_in,
                        examples_per_page=4, label='Sample garments')
        with gr.Column():
            result_out = gr.Image(label='Result', height=420)
            mask_out   = gr.Image(label='Clothing area', height=200)
            status_out = gr.Textbox(label='Status', interactive=False)
    desc_in = gr.Textbox(label='Garment description',
                         placeholder='e.g. white cotton t-shirt', value='a shirt')
    auto_mask_cb = gr.Checkbox(label='Auto-detect clothing area', value=True)
    with gr.Accordion('Advanced', open=False):
        steps_sl = gr.Slider(15,40,value=30,step=1,label='Denoising steps')
        seed_nb  = gr.Number(value=42,label='Seed',precision=0)
    run_btn = gr.Button('\u2728 Try It On', variant='primary', size='lg')
    run_btn.click(fn=run_tryon,
                  inputs=[person_in,garment_in,desc_in,auto_mask_cb,steps_sl,seed_nb],
                  outputs=[result_out,mask_out,status_out])

demo.launch(share=True, debug=True)
